In [1]:
import pandas as pd

In [2]:
# Requires the restricted OPTN extract at data/restricted/kidney_waitlist_analytic.csv.gz.
# Download it from the Drive link in the fellows data-access email (see
# Challenge-Project-Overview.md). The folder is gitignored, so it won't come with a clone.
df = pd.read_csv("../data/restricted/kidney_waitlist_analytic.csv.gz")

C:\Users\gorge\AppData\Local\Temp\ipykernel_27756\3605479730.py:4: DtypeWarning: Columns (0: PREV_TX, 1: TX_DATE, 2: DON_TY, 3: MULTIORG, 4: ORGAN, 5: TRR_ID_CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/restricted/kidney_waitlist_analytic.csv.gz")


In [3]:
# find and sum all null columns
df.isna().sum()

ON_DIALYSIS                  0
A2A2B_ELIGIBILITY       456604
GENDER                       0
ABO                          0
BMI_TCR                   1558
FUNC_STAT_TCR             3975
INIT_STAT                    0
INIT_CPRA               142741
END_CPRA                142448
REM_CD                  103300
DAYSWAIT_CHRON              53
END_STAT                     0
INIT_AGE                     0
DIALYSIS_DATE           134245
END_DATE                     0
INIT_DATE                    0
ETHCAT                       0
PT_CODE                      0
DAYSWAIT_ALLOC            6745
COMPOSITE_DEATH_DATE    398512
REGION                       0
WL_ID_CODE                   0
PREV_TX                 264107
TX_DATE                 264107
DON_TY                  264107
DIAG_KI                 265540
MULTIORG                483004
ORGAN                   264107
PSTATUS                 264107
PTIME                   267350
TRR_ID_CODE             264107
DONOR_ID                264107
LISTING_

In [4]:
# see the data type of each column
df.dtypes

ON_DIALYSIS                 str
A2A2B_ELIGIBILITY       float64
GENDER                      str
ABO                         str
BMI_TCR                 float64
FUNC_STAT_TCR           float64
INIT_STAT                 int64
INIT_CPRA               float64
END_CPRA                float64
REM_CD                  float64
DAYSWAIT_CHRON          float64
END_STAT                  int64
INIT_AGE                  int64
DIALYSIS_DATE               str
END_DATE                    str
INIT_DATE                   str
ETHCAT                    int64
PT_CODE                   int64
DAYSWAIT_ALLOC          float64
COMPOSITE_DEATH_DATE        str
REGION                    int64
WL_ID_CODE                int64
PREV_TX                     str
TX_DATE                     str
DON_TY                      str
DIAG_KI                 float64
MULTIORG                    str
ORGAN                       str
PSTATUS                 float64
PTIME                   float64
TRR_ID_CODE                 str
DONOR_ID

In [ ]:
# we want to remember how many rows we started with, so we can see how much data we have lost
rows_started_with = len(df)

# we want to turn the date columns into real dates
# errors = 'coerce' means invalid/unparseable values are converted in NaN/NaT

to_date = ['INIT_DATE', 'TX_DATE', 'END_DATE']
for value in to_date:
    df[value] = pd.to_datetime(df[value], errors='coerce')

# mark who gets a transplant
# patient with a date gets TRUE, patient without gets FALSE
df['TRANSPLANTED'] = df['TX_DATE'].notna()

# calculate how long the patient waited
# patients with no transplant date get a blank
df['DAYS_TO_EVENT'] = (df['TX_DATE'] - df['INIT_DATE']).dt.days

# the first part asks if the transplant date is earlier than the listing date (NOT POSSIBLE!) If the answer is true, the data is wrong
# the second part asks a similar question. can't leave the waitlist before joining it
# need or: a row is bad if first part, second part, or both are true
bad_data = (df['TX_DATE'] < df['INIT_DATE']) | (df['END_DATE'] < df['INIT_DATE'])

# keep rows with data that makes sense
# makes a fresh table and reassigns it to df
df = df[~bad_data].copy()

# print a summary of what happens
print(f"started: {rows_started_with:,} | transplanted: {transplanted_rows:,} | "
      f"bad dates removed: {bad_data.sum():,} | final: {len(df):,}")


started: 494,862 | transplanted: 230,755 | bad dates removed: 0 | final: 230,755


In [6]:
# drop columns that tell the model the answer
# TX_DATE, DON_TY, DONOR_ID, ORGAN, TRR_ID_CODE only get populated when a transplant happens
# REM_CD, END_DATE, END_STAT, END_CPRA describe status after leaving waitlist
# COMPOSITE_DEATH_DATE only has a val if the patient dies
# DAYSWAIT_CHRON, DAYSWAIT_ALLOC same nums as the ones we want to predict
# PTIME, PSTATUS post transplant
# Basically, everything populated after the transplant happens has to be removed
drop_col = [
    'REM_CD', 'END_DATE', 'TX_DATE', 'COMPOSITE_DEATH_DATE',
    'END_STAT', 'END_CPRA', 'DAYSWAIT_CHRON', 'DAYSWAIT_ALLOC',
    'DON_TY', 'ORGAN', 'TRR_ID_CODE', 'DONOR_ID', 'PTIME', 'PSTATUS'
]
# safety check for columns that don't exist. if the value exists, drop it
df_features = df.drop(columns=[col for col in drop_col if col in df.columns])

dropped_columns = [col for col in drop_col if col in df_features.columns]
if(dropped_columns): # if this is true, columns were not dropped properly
    print("Columns were not dropped properly!")
else:
    print("Columns don't exist anymore.")

Columns don't exist anymore.


In [7]:
# 1. Fill MULTIORG missing values
df_features['MULTIORG'] = df_features['MULTIORG'].fillna('N')

# 2. Select object-type columns correctly
string_cols_list = df_features.select_dtypes(include=['object']).columns.tolist()

# Exclude DIALYSIS_DATE so it can be parsed as a datetime later
string_col_not_dialysis = [
    col for col in string_cols_list if col != 'DIALYSIS_DATE'
]
df_features[string_col_not_dialysis] = df_features[
    string_col_not_dialysis
].fillna('Unknown')

# 3. Verify that non-date string columns have 0 missing values
print(
    'Missing values in object columns (excluding DIALYSIS_DATE):',
    df_features[string_col_not_dialysis].isna().sum().sum(),
)

Missing values in object columns (excluding DIALYSIS_DATE): 0


C:\Users\gorge\AppData\Local\Temp\ipykernel_27756\2198478976.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols_list = df_features.select_dtypes(include=['object']).columns.tolist()


In [ ]:
# fill numeric (int and float) type columns with missing values

# continuous fields: fill them with the median. don't put dummy values like 0, which would corrupt the data. distribution likely skewed, so use median over mean
continuous_num = ['BMI_TCR', 'INIT_CPRA']
df_features[continuous_num] = df_features[continuous_num].fillna(df_features[continuous_num].median())

print(df_features['BMI_TCR'].isna().sum())
df_features['INIT_CPRA_MISSING'] = df_features['INIT_CPRA'].isna().astype('int8')
print(df_features['INIT_CPRA'].isna().sum())

# fill DIAG_KI with a sentinel that's an invalid number
df_features['DIAG_KI'] = df_features['DIAG_KI'].fillna(-1)
# convert it to a string object, so that it's not treated as a number
df_features['DIAG_KI'] = df_features['DIAG_KI'].astype('str')

# fill FUNC_STAT_TCR with a sentinel that's an invalid number
df_features['FUNC_STAT_TCR'] = df_features['FUNC_STAT_TCR'].fillna(-1)
# convert it to a string object, so that it's not treated as a number
df_features['FUNC_STAT_TCR'] = df_features['FUNC_STAT_TCR'].astype('str')

df_features['A2A2B_ELIGIBILITY'] = df_features['A2A2B_ELIGIBILITY'].fillna(-1)
df_features['A2A2B_ELIGIBILITY'] = df_features['A2A2B_ELIGIBILITY'].astype('str')

print(df_features['DIALYSIS_DATE'].isna().sum())
print(df_features['MULTIORG'].isna().sum())
print(df_features['DIAG_KI'].isna().sum())
print(df_features['A2A2B_ELIGIBILITY'].isna().sum())

0
0
61519
0
0
0


In [9]:
# make sure they're all null
df_features.isna().sum()

ON_DIALYSIS              0
A2A2B_ELIGIBILITY        0
GENDER                   0
ABO                      0
BMI_TCR                  0
FUNC_STAT_TCR            0
INIT_STAT                0
INIT_CPRA                0
INIT_AGE                 0
DIALYSIS_DATE        61519
INIT_DATE                0
ETHCAT                   0
PT_CODE                  0
REGION                   0
WL_ID_CODE               0
PREV_TX                  0
DIAG_KI                  0
MULTIORG                 0
LISTING_CTR_CODE         0
outcome                  0
event_adverse            0
event_transplant         0
censored                 0
days_to_event            0
TRANSPLANTED             0
DAYS_TO_EVENT            0
dtype: int64

In [10]:
# find outliers in numeric-type columns and winsorize using IQR
to_winscorize = ['BMI_TCR']
# INIT_CPRA reacted badly to IQR, so leave it alone. it's a percentage, so the values cannot really be outliers
# INIT_AGE also reacted badly (cut away valid ages)
# DON'T WINSCORIZE DAYS_TO_EVENT--CUTTING AWAY OUTLEIRS CUTS OUT DATA!!!

# IQR (retains data size and reduces distortion)
for col in to_winscorize:
    Q1 = df_features[col].quantile(0.25)
    Q3 = df_features[col].quantile(0.75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_features[(df_features[col] < lower_bound) | (df_features[col] > upper_bound)][col]
    print(f"{col}: {len(outliers)} outliers, range [{outliers.min()}, {outliers.max()}], bounds [{lower_bound:.1f}, {upper_bound:.1f}]")
    df_features[col] = df_features[col].clip(lower=lower_bound, upper=upper_bound)


BMI_TCR: 1164 outliers, range [0.18, 430226.67], bounds [11.8, 44.8]


In [11]:
# convert strings to datetime to do dialysis subtraction
df_features['INIT_DATE'] = pd.to_datetime(df_features['INIT_DATE'])
df_features['INIT_DATE'].dtype

df_features['DIALYSIS_DATE'] = pd.to_datetime(df_features['DIALYSIS_DATE'])
df_features['DIALYSIS_DATE'].dtype

# find length of dialysis before listing
df_features['DIALYSIS_DURATION'] = (df_features['INIT_DATE'] - df_features['DIALYSIS_DATE']).dt.days

# if null value, replace with 0 (it means values were missing and never on dialysis)
df_features['DIALYSIS_DURATION'] = df_features['DIALYSIS_DURATION'].fillna(0)

#/
# Negative values in dialysis likely mean the listing was preemptative,
# but you can't have negative time. Clip to 0
# /#
df_features['DIALYSIS_AFTER_LISTING'] = (df_features['DIALYSIS_DURATION'] < 0).astype('int8')
# limit min values to 0
df_features['DIALYSIS_DURATION'] = df_features['DIALYSIS_DURATION'].clip(lower=0)

# drop raw date column as it's not necessary any longer
df_features = df_features.drop(columns=['DIALYSIS_DATE'])

# both should be 0
print(df_features['DIALYSIS_DURATION'].isna().sum())
print((df_features['DIALYSIS_DURATION'] < 0).sum())   

0
0


In [12]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_features.to_csv("../data/processed/cleaned_waitlist.csv", index=False)